# Debugger — inspect cluster smoke output

Investigates the UNI-66 Llama smoke results from Snellius across all four prompt conditions:

| `CONDITION` | File | `relations` keys |
|---|---|---|
| `"causal"` | `tma_subset.smoke3_relations.jsonl` | `causal_relations` |
| `"temporal"` | `tma_subset.smoke3_relations.temporal.jsonl` | `temporal_relations` |
| `"tcindep"` | `tma_subset.smoke3_relations.tcindep.jsonl` | `temporal_relations` + `causal_relations` |
| `"tcjoint"` | `tma_subset.smoke3_relations.tcjoint.jsonl` | `joint_relations` |

Change the `CONDITION` constant in cell 1 and Run All Cells to switch between them. All downstream cells iterate over `row['relations'].items()`, so they handle any of the four schemas automatically.

**Hypotheses being inspected:**
1. Llama is only linking *consecutive* events (eID gap == 1).
2. Label distribution collapses to a single label (e.g. `CAUSE` in `causal`, `BEFORE` in `temporal`).
3. Cross-sentence relations are rare.
4. Coverage: large fraction of detected events are not referenced by any relation.


In [1]:
CONDITION = "tcjoint"   # change to: "causal" | "temporal" | "tcindep" | "tcjoint"

import json, sys
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SUFFIX_BY_CONDITION = {
    "causal":   "",                # original file has no suffix
    "temporal": ".temporal",
    "tcindep":  ".tcindep",
    "tcjoint":  ".tcjoint",
}
REL_PATH = ROOT / "data" / "intermediate" / f"tma_subset.smoke3_relations{SUFFIX_BY_CONDITION[CONDITION]}.jsonl"

sys.path.insert(0, str(ROOT))
from models.llama.inference import inline_events

print(f"CONDITION = {CONDITION}")
print(f"REL_PATH  = {REL_PATH}")
print(f"exists    = {REL_PATH.exists()}")

rows = [json.loads(line) for line in REL_PATH.read_text().splitlines()]
print(f"rows in file: {len(rows)}")


CONDITION = tcjoint
REL_PATH  = /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis/data/intermediate/tma_subset.smoke3_relations.tcjoint.jsonl
exists    = True
rows in file: 2


## 1. Big-picture overview

In [2]:
print(f'top-level keys (row 0): {list(rows[0].keys())}')
print(f'relations keys (row 0): {list(rows[0]["relations"].keys())}')

pd.DataFrame([
    {
        'summary_id': r.get('summary_id'),
        'n_sentences': r.get('n_sentences'),
        'n_tokens': r.get('n_tokens'),
        'n_events': len(r['events']),
        'n_relations': sum(len(v) for v in r['relations'].values()),
    }
    for r in rows
])

top-level keys (row 0): ['wikidata_id', 'summary_id', 'lang', 'text', 'sentences', 'n_sentences', 'n_tokens', 'genres', 'split', 'events', 'relations']
relations keys (row 0): ['joint_relations']


,summary_id,n_sentences,n_tokens,n_events,n_relations
0,de,28,743,62,14
1,en,13,525,35,6


## 2. Inline view — annotated summary + relations JSON

Shows each summary as Llama actually saw it: event markers `[eID|trigger|TYPE]` spliced into the text at event spans (the same `inline_events()` helper the pipeline uses). Below each annotated summary, the raw `relations` JSON Llama produced. This is the most natural way to read the output — every `source`/`target` eID in the JSON is visible as a marker inline above.

In [3]:
from collections import defaultdict
from IPython.display import display, Markdown

for i, row in enumerate(rows):
      by_sent = defaultdict(list)
      for ev in row['events']:
          by_sent[ev['sent_id']].append(ev)

      md = [
          f"### row {i} — `{row.get('summary_id','?')}` "
          f"(events={len(row['events'])}, "
          f"relations={sum(len(v) for v in row['relations'].values())})",
          "",
          "**Annotated summary** (events inlined as `[eID|trigger|TYPE]`, bolded):",
          "",
      ]

      for sent_id, sent in enumerate(row['sentences']):
          evs = sorted(by_sent.get(sent_id, []), key=lambda e: e['start'])
          out, cursor = [], 0
          for ev in evs:
              out.append(sent[cursor:ev['start']])
              out.append(f"**[{ev['event_id']}|{ev['trigger']}|{ev['event_type']}]**")
              cursor = ev['end']
          out.append(sent[cursor:])
          md.append(f"- s{sent_id}: {''.join(out)}")

      md += [
          "",
          "**Relations (raw JSON):**",
          "",
          "```json",
          json.dumps(row['relations'], indent=2),
          "```",
      ]
      display(Markdown("\n".join(md)))

### row 0 — `de` (events=62, relations=14)

**Annotated summary** (events inlined as `[eID|trigger|TYPE]`, bolded):

- s0: New York: As a 12-year-old boy, Jacob, longing for his missing father, John Reckless, **[e1|enters|Arriving]** his room.
- s1: There, he **[e2|finds|Know]** a message that takes him through a mirror into a world of mirrors.
- s2: Where the characters of Grimm's Fairy Tales actually **[e3|exist|Presence]**, but the world has **[e4|evolved|Coming_to_be]** and **[e5|modernized|Cause_to_make_progress]** and is now in a state of decay.
- s3: From now on, he often **[e6|visits|Traveling]** the mirror world secretly, at fifteen he **[e7|sneak|Self_motion]**s away for the first time for weeks.
- s4: In his increasingly long periods of absence from his world, he **[e8|becomes|Becoming]** an apprentice to the treasure hunter Chanute.
- s5: Later, Jacob himself **[e9|becomes|Becoming]** a treasure hunter for the Empress of Austria.
- s6: Twelve years after Jacob Reckless first **[e10|entered|Arriving]** the world behind the mirror, his brother Will sees him disappear into the magic mirror and follows him.
- s7: There, Will is **[e11|attacked|Attack]** by a goyl, which **[e12|causes|Causation]** him to slowly grow a skin of jade that **[e13|transforms|Change]** him into a goyl.
- s8: Will's friend Clara follows him through the mirror and now **[e14|sets off|Departing]** with him and Jacob and his companion Fox, who are trying to **[e15|find|Know]** an antidote to the curse.
- s9: The first attempt to cure Will Reckless with berries from the stump of a child-eating witch from the Black Forest fails, Jacob and Fox narrowly **[e16|escape|Escaping]** the so-called "Schneider".
- s10: The tailor **[e17|makes|Manufacturing]** clothes out of human skin.
- s11: At the same time, the Goyl under King Kamen and with the **[e18|help|Assistance]** of the Dark Fairy are waging war against the Imperial armies, and many people have already fallen victim to the war.
- s12: The Dark Fairy has a dream of a Jade Goyl, a legendary guardian in fairy tales, **[e19|sent|Sending]** to **[e20|save|Rescuing]** the Goyl in her darkest hour.
- s13: And so the Goyl Hentzau goes with an **[e21|armed|Bearing_arms]** troop in **[e22|search|Scrutiny]** of Will Reckless, the only one to grow the jade skin.
- s14: Meanwhile, Jacob and the others make their way to the Red Fairy Miranda, from whom he hopes to **[e23|get|Getting]** **[e24|help|Assistance]**.
- s15: But only the dwarf Evenaugh Valiant **[e25|knows|Know]** the way there, and so he must trust him whether he likes it or not, even though the dwarf has betrayed him before.
- s16: The Red Fairy **[e26|promises|Commitment]** to **[e27|help|Assistance]** Jacob, but only on the condition that she **[e28|destroy|Destroying]** her sister, the Dark Fairy.
- s17: When he **[e29|returns|Arriving]** from the fairy, his brother is **[e30|kidnapped|Kidnapping]** by Hentzau and so he must follow him underground with Valiant to free Will.
- s18: But the attempt fails, and they must **[e31|flee|Escaping]**.
- s19: In order to **[e32|restore|Recovering]** peace, Empress Thérèse of Austria and King Kamen **[e33|agree|Agree_or_refuse_to_act]** to his soon **[e34|marriage|Forming_relationships]** to Thérèse's daughter.
- s20: Will, who has now completely **[e35|transformed|Change]** into a goyl and **[e36|forgotten|GiveUp]** his former life, is the king's bodyguard.
- s21: Shortly before the **[e37|wedding|Social_event]**, Jacob manages to **[e38|capture|Conquering]** the Dark Fairy by pronoun**[e39|cing|Statement]** her name, which he learned from Miranda.
- s22: Now he's trying to find Will and **[e40|bring|Bringing]** him to the Dark Fairy so she can **[e41|turn|Change]** him back.
- s23: The Empress **[e42|learns|Know]** that the Dark Fairy no longer **[e43|protects|Defending]** Kamen and tries to have him and the Goyls **[e44|present|Presence]** at the **[e45|wedding|Social_event]**, including Will, **[e46|killed|Killing]**.
- s24: In order to **[e47|save|Rescuing]** his brother, Jacob is forced to **[e48|free|Releasing]** the Dark Fairy, who then **[e49|appears|Presence]** and **[e50|kills|Killing]** most of the people **[e51|present|Presence]**.
- s25: The Goyls will **[e52|take|Conquering]** the rest as hostages.
- s26: Amazingly, the Dark Fairy **[e53|turns|Becoming]** Will back into a human, but **[e54|threatens|Warning]** to **[e55|kill|Killing]** him if Jacob doesn't take him far away.
- s27: Will and Clara **[e56|return|Arriving]** to the other world, but Jacob Reckless is **[e57|cursed|Judgment_communication]** by the Dark Fairy when he **[e58|pro|Statement]**no**[e59|unce|Statement]**s her name and **[e60|sets|Placing]** out to **[e61|find|Know]** an antidote, otherwise he will **[e62|die|Death]** within a year.

**Relations (raw JSON):**

```json
{
  "joint_relations": [
    {
      "source": "e12",
      "target": "e11",
      "relation": "CAUSE_TO_END_BEFORE"
    },
    {
      "source": "e17",
      "target": "e16",
      "relation": "ENABLE_BEFORE"
    },
    {
      "source": "e18",
      "target": "e19",
      "relation": "CAUSE_BEFORE"
    },
    {
      "source": "e21",
      "target": "e22",
      "relation": "CAUSE_BEFORE"
    },
    {
      "source": "e25",
      "target": "e23",
      "relation": "ENABLE_BEFORE"
    },
    {
      "source": "e26",
      "target": "e27",
      "relation": "CAUSE_BEFORE"
    },
    {
      "source": "e28",
      "target": "e29",
      "relation": "CAUSE_BEFORE"
    },
    {
      "source": "e31",
      "target": "e30",
      "relation": "PREVENT_BEFORE"
    },
    {
      "source": "e38",
      "target": "e39",
      "relation": "CAUSE_TO_END_BEFORE"
    },
    {
      "source": "e42",
      "target": "e43",
      "relation": "CAUSE_BEFORE"
    },
    {
      "source": "e47",
      "target": "e48",
      "relation": "CAUSE_BEFORE"
    },
    {
      "source": "e49",
      "target": "e50",
      "relation": "CAUSE_TO_END_BEFORE"
    },
    {
      "source": "e53",
      "target": "e54",
      "relation": "CAUSE_BEFORE"
    },
    {
      "source": "e58",
      "target": "e59",
      "relation": "CAUSE_TO_END_BEFORE"
    }
  ]
}
```

### row 1 — `en` (events=35, relations=6)

**Annotated summary** (events inlined as `[eID|trigger|TYPE]`, bolded):

- s0: In late 19th century Mexico, Federales **[e1|capture|Conquering]** Quintero (Fernando Rey), a **[e2|revolutionary|Change_of_leadership]** who attempts to rally those **[e3|opposing|Agree_or_refuse_to_act]** the dictatorship of President Díaz.
- s1: Before **[e4|going|Self_motion]** **[e5|to|Motion]** prison, Quintero **[e6|gives|Giving]** his lieutenant, Maximiliano O'Leary (Reni Santoni), $600 (equivalent to $20,000 in 2022) with which to continue the cause.
- s2: Bandit chief Carlos Lobero (Frank Silvera) **[e7|demands|Request]** that the money be **[e8|used|Using]** for guns and ammunition, but Max instead **[e9|crosses|Motion_directional]** the border in search of Chris Adams (George Kennedy): a legendary, American gunman whom his cousin had **[e10|told|Telling]** him about.
- s3: Max finally **[e11|finds|Know]** the laconic Chris, witnessing him free a man from a rigged trial, first by **[e12|using|Using]** his wits, then with the famed hair-trigger skill as a gunfighter.
- s4: Chris **[e13|agrees|Agree_or_refuse_to_act]** to mount a **[e14|rescue|Rescuing]** of Quintero and **[e15|uses|Using]** $500 of Max's money to recruit five highly trained combatants: Keno (Monte Markham), a horse thief and hand-to-hand combat expert (whom Chris **[e16|saved|Rescuing]** from hanging); Cassie (Bernie Casey), a brawny but intelligent former slave, who can handle dynamite; Slater (Joe Don Baker), a one-armed, sideshow sharp-shootist; a tubercular wrangler **[e17|called|Name_conferral]** "P.J."
- s5: (Scott Thomas), and Levi Morgan (James Whitmore), an aging family man who is doubtful of his worth, despite his incredible knife-throwing skills.
- s6: En route to Mexico, the motley band of Americans **[e18|becomes|Becoming]** less mercenary when **[e19|observing|Perception_active]** the brutal treatment of the peasants.
- s7: Their journey is **[e20|marked|Recording]** by **[e21|encounters|Hostile_encounter]** with a political prisoner's little boy, Emiliano Zapata (Tony Davis) and a pretty peasant girl, Tina (Wende Wagner), who falls in love with P.J. When Lobero learns that Max did not **[e22|buy|Commerce_buy]** guns with the $600, he **[e23|refuses|Agree_or_refuse_to_act]** to **[e24|allow|Preventing_or_letting]** his men to take part in Quintero's **[e25|rescue|Rescuing]**.
- s8: Realizing that he needs **[e26|support|Supporting]**, Chris **[e27|free|Releasing]**s a prison gang that includes Zapata's father, then **[e28|trains|Education_teaching]** them in military tactics.
- s9: Despite their superior fighting skills and strategy, Chris' men are outnumbered and their valiant effort to free Quintero **[e29|appears|Presence]** doomed.
- s10: At the last moment, 50 of Lobero's bandits, having **[e30|slain|Killing]** their leader for his lack of patriotism, thunder onto the prison grounds and **[e31|turn|Change]** the tide of **[e32|battle|Hostile_encounter]**.
- s11: Of the original seven, only Chris, Max and Levi survive.
- s12: Before **[e33|riding|Self_motion]** home, Chris and Levi **[e34|leave|Departing]** behind the $600 the peasants had **[e35|collected|Come_together]**.

**Relations (raw JSON):**

```json
{
  "joint_relations": [
    {
      "source": "e6",
      "target": "e7",
      "relation": "CAUSE_BEFORE"
    },
    {
      "source": "e9",
      "target": "e10",
      "relation": "CAUSE_BEFORE"
    },
    {
      "source": "e12",
      "target": "e13",
      "relation": "CAUSE_BEFORE"
    },
    {
      "source": "e16",
      "target": "e17",
      "relation": "CAUSE_BEFORE"
    },
    {
      "source": "e27",
      "target": "e28",
      "relation": "ENABLE_BEFORE"
    },
    {
      "source": "e30",
      "target": "e31",
      "relation": "CAUSE_TO_END_BEFORE"
    }
  ]
}
```

## 3. eID-distance distribution (the hypothesis check)

For each relation, compute `|source_id - target_id|`. If Llama only links consecutive events, every diff = 1.

In [4]:
all_diffs = []
for row in rows:
    for rels in row['relations'].values():
        for r in rels:
            all_diffs.append(abs(int(r['source'][1:]) - int(r['target'][1:])))

print(f'total relations across all rows: {len(all_diffs)}')
print(f'diff distribution: {sorted(Counter(all_diffs).items())}')
if all_diffs:
    consecutive = sum(1 for d in all_diffs if d == 1)
    print(f'consecutive (diff==1): {consecutive}/{len(all_diffs)} = {consecutive/len(all_diffs):.1%}')
    print(f'max diff: {max(all_diffs)}')
    print(f'mean diff: {sum(all_diffs)/len(all_diffs):.2f}')

total relations across all rows: 20
diff distribution: [(1, 19), (2, 1)]
consecutive (diff==1): 19/20 = 95.0%
max diff: 2
mean diff: 1.05


## 4. Per-row relations with triggers resolved (arrow view)

Same data as §2 but compressed: each relation rendered as `source (trigger) --LABEL--> target (trigger)`, with the eID gap.

In [5]:
for i, row in enumerate(rows):
    ev = {e['event_id']: e for e in row['events']}
    print(f"=== row {i}  summary_id={row.get('summary_id','?')}  events={len(row['events'])} ===")
    for rel_type, rels in row['relations'].items():
        print(f'  [{rel_type}]  n={len(rels)}')
        for r in rels:
            s, t = ev.get(r['source']), ev.get(r['target'])
            s_repr = f"{r['source']} ({s['trigger']!r}, sent {s['sent_id']})" if s else f"{r['source']} [MISSING]"
            t_repr = f"{r['target']} ({t['trigger']!r}, sent {t['sent_id']})" if t else f"{r['target']} [MISSING]"
            gap = abs(int(r['source'][1:]) - int(r['target'][1:]))
            print(f'    {s_repr}  --{r["relation"]}-->  {t_repr}   (eID gap = {gap})')
    print()

=== row 0  summary_id=de  events=62 ===
  [joint_relations]  n=14
    e12 ('causes', sent 7)  --CAUSE_TO_END_BEFORE-->  e11 ('attacked', sent 7)   (eID gap = 1)
    e17 ('makes', sent 10)  --ENABLE_BEFORE-->  e16 ('escape', sent 9)   (eID gap = 1)
    e18 ('help', sent 11)  --CAUSE_BEFORE-->  e19 ('sent', sent 12)   (eID gap = 1)
    e21 ('armed', sent 13)  --CAUSE_BEFORE-->  e22 ('search', sent 13)   (eID gap = 1)
    e25 ('knows', sent 15)  --ENABLE_BEFORE-->  e23 ('get', sent 14)   (eID gap = 2)
    e26 ('promises', sent 16)  --CAUSE_BEFORE-->  e27 ('help', sent 16)   (eID gap = 1)
    e28 ('destroy', sent 16)  --CAUSE_BEFORE-->  e29 ('returns', sent 17)   (eID gap = 1)
    e31 ('flee', sent 18)  --PREVENT_BEFORE-->  e30 ('kidnapped', sent 17)   (eID gap = 1)
    e38 ('capture', sent 21)  --CAUSE_TO_END_BEFORE-->  e39 ('cing', sent 21)   (eID gap = 1)
    e42 ('learns', sent 23)  --CAUSE_BEFORE-->  e43 ('protects', sent 23)   (eID gap = 1)
    e47 ('save', sent 24)  --CAUSE_BEFORE--

## 5. eID coverage

Which events actually appear in any relation? Are there events Llama ignored entirely? Any hallucinated eIDs (mentioned in `relations` but not in `events`)?

In [6]:
for i, row in enumerate(rows):
    all_eids = {e['event_id'] for e in row['events']}
    used_eids = set()
    for rels in row['relations'].values():
        for r in rels:
            used_eids.add(r['source'])
            used_eids.add(r['target'])
    unused = all_eids - used_eids
    invalid = used_eids - all_eids
    print(f'row {i}: {len(used_eids & all_eids)}/{len(all_eids)} events appear in relations '
          f'({len(used_eids & all_eids)/len(all_eids):.0%} coverage); '
          f'{len(unused)} unused; {len(invalid)} hallucinated')
    if invalid:
        print(f'  ⚠ hallucinated eIDs: {sorted(invalid)}')

row 0: 28/62 events appear in relations (45% coverage); 34 unused; 0 hallucinated
row 1: 12/35 events appear in relations (34% coverage); 23 unused; 0 hallucinated


## 6. n=200 audit — failure-mode inspection (UNI-24 pilot review)

The four `llama-*-n200-*.out` logs in `data/intermediate/llama_logs/` are the four-condition pilot batch from [UNI-52](https://linear.app/uva-school-thesis/issue/UNI-52/). These cells parse them, join with the input rows from `tma_subset_events_full.jsonl`, and inspect the stories that triggered each failure mode — so we can iterate on the prompts in [UNI-24](https://linear.app/uva-school-thesis/issue/UNI-24/) deliberately rather than blindly.

In [7]:
# === n=200 audit findings (UNI-24 pilot inspection) ===
# Parses the four llama-*-n200-*.out logs + joins with the input rows from
# tma_subset_events_full.jsonl so we can inspect WHICH stories triggered each
# failure category. Independent of the smoke3 CONDITION selector above.
import re

LOGS_DIR    = ROOT / "data" / "intermediate" / "llama_logs"
EVENTS_PATH = ROOT / "data" / "intermediate" / "tma_subset_events_full.jsonl"
JOB_NAME_TO_COND = {
    "temporal": "temporal",
    "causal":   "causal",
    "tcindep":  "temporal_causal_independent",
    "tcjoint":  "temporal_causal_joint",
}
SUCCESS_RE = re.compile(r"^row (\d+): input_tokens=(\d+) output_tokens=(\d+)\s*$")
ERROR_RE   = re.compile(r"^row (\d+): (?!input_tokens=)(.+)$")

def classify_error(msg: str) -> str:
    if "Unterminated string" in msg or msg.startswith("Unterminated"): return "truncation"
    if msg.startswith("Expecting value"):                              return "prose_wrap"
    if msg.startswith("unknown label"):                                return "label_hallucination"
    if msg.startswith("bad eID"):                                      return "bad_eid"
    if "Expecting property name" in msg or ("Expecting" in msg and "delimiter" in msg):
        return "json_malformed"
    return "other"

# First 200 rows of the events file = exactly what the audit jobs processed.
input_rows = []
with EVENTS_PATH.open() as f:
    for i, line in enumerate(f):
        if i >= 200: break
        input_rows.append(json.loads(line))

records = []
for log_path in sorted(LOGS_DIR.glob("llama-*-n200-*.out")):
    suffix = log_path.name.split("-")[1]
    cond = JOB_NAME_TO_COND.get(suffix)
    if cond is None: continue
    per_row: dict[int, dict] = {}
    for line in log_path.read_text().splitlines():
        if (m := SUCCESS_RE.match(line)):
            per_row[int(m.group(1))] = {
                "input_tokens":  int(m.group(2)),
                "output_tokens": int(m.group(3)),
                "error":         None,
            }
        elif (m := ERROR_RE.match(line)):
            ridx = int(m.group(1))
            if ridx in per_row:
                per_row[ridx]["error"] = m.group(2).strip()
            else:
                per_row[ridx] = {"input_tokens": None, "output_tokens": None, "error": m.group(2).strip()}
    for ridx, info in sorted(per_row.items()):
        records.append({"condition": cond, "row_idx": ridx, **info})

audit_df = pd.DataFrame(records)
audit_df["category"] = audit_df["error"].apply(lambda e: classify_error(e) if isinstance(e, str) else "ok")
audit_df["n_events"] = audit_df["row_idx"].map(lambda i: len(input_rows[i]["events"]))
audit_df["n_words"]  = audit_df["row_idx"].map(lambda i: len(input_rows[i]["text"].split()))

# Per-condition × failure-category counts + total fail %.
counts = audit_df.groupby(["condition", "category"]).size().unstack(fill_value=0)
counts["total"]    = audit_df.groupby("condition").size()
counts["fail_pct"] = (100 * (counts["total"] - counts.get("ok", 0)) / counts["total"]).round(1)
print(f"\033[1;33mn=200 audit — failure breakdown per condition:\033[0m\n")
display(counts)

n=200 audit — failure breakdown per condition:



category,bad_eid,json_malformed,label_hallucination,ok,prose_wrap,truncation,total,fail_pct
condition,,,,,,,,
causal,1,6,1,177,11,4,200,11.5
temporal,0,5,9,154,4,28,200,23.0
temporal_causal_independent,0,14,48,54,30,54,200,73.0
temporal_causal_joint,0,1,11,164,20,4,200,18.0


### 6.1 Truncation (output hit the static 2,048 cap → Unterminated string)

These should mostly disappear after the dynamic `max_new_tokens` fix shipped with UNI-52. Inspecting them here mainly tells us *which stories* are output-heavy, so we know what to expect under the dynamic cap.

In [8]:
# Cases where the LLM hit the static max_new_tokens=2048 cap and the resulting JSON was
# cut mid-string. These should mostly disappear after the dynamic max_new_tokens fix in
# infer_relations.py (UNI-52 close-out). Worth checking which story sizes triggered them.
trunc = audit_df[audit_df["category"] == "truncation"].sort_values("n_events", ascending=False)
print(f"{len(trunc)} truncation failures across conditions:")
print(trunc.groupby("condition").size().to_string())
print()
display(trunc[["condition", "row_idx", "output_tokens", "n_events", "n_words", "error"]].head(15))

# One concrete example with the input story.
if len(trunc):
    ex = trunc.iloc[0]
    r  = input_rows[int(ex["row_idx"])]
    display(Markdown(
        f"### Example truncation\n"
        f"- condition: `{ex['condition']}`\n"
        f"- row_idx: `{ex['row_idx']}` (wikidata_id `{r['wikidata_id']}`, summary_id `{r['summary_id']}`)\n"
        f"- n_events: **{len(r['events'])}**, n_words: **{len(r['text'].split())}**\n"
        f"- output_tokens: **{ex['output_tokens']}** (cap)\n"
        f"- error: `{ex['error']}`\n"
    ))

90 truncation failures across conditions:
condition
causal                          4
temporal                       28
temporal_causal_independent    54
temporal_causal_joint           4



,condition,row_idx,output_tokens,n_events,n_words,error
151,causal,151,2048,181,1647,Unterminated string starting at: line 366 colu...
551,temporal_causal_joint,151,2048,181,1647,Unterminated string starting at: line 355 colu...
759,temporal,159,2048,160,314,Unterminated string starting at: line 95 colum...
0,causal,0,2048,155,1135,Unterminated string starting at: line 366 colu...
600,temporal,0,2048,155,1135,Unterminated string starting at: line 95 colum...
753,temporal,153,2048,142,927,Unterminated string starting at: line 95 colum...
765,temporal,165,2048,134,22,Unterminated string starting at: line 95 colum...
304,temporal_causal_independent,104,2048,127,820,Unterminated string starting at: line 95 colum...
704,temporal,104,2048,127,820,Unterminated string starting at: line 95 colum...
651,temporal,51,2048,124,1101,Unterminated string starting at: line 95 colum...


### Example truncation
- condition: `causal`
- row_idx: `151` (wikidata_id `102438`, summary_id `fr`)
- n_events: **181**, n_words: **1647**
- output_tokens: **2048** (cap)
- error: `Unterminated string starting at: line 366 column 17 (char 5476)`


### 6.2 Label hallucination (LLM invented a label outside the codebook)

E.g. `unknown label: AFTER` when the temporal codebook only has BEFORE / OVERLAPS / CONTAINS / IDENTITY. The biggest concentration is in `temporal_causal_independent` (48 failures) — that condition's prompt may not constrain the LLM tightly enough.

In [9]:
# Cases where parse_and_validate said "unknown label". Some are real hallucinations
# (label is in NEITHER codebook), but some — especially in temporal_causal_independent —
# are SECTION-MISPLACEMENT: a valid causal label (e.g. CAUSE_TO_END, CAUSE) appearing
# under the "temporal_relations" array (or vice versa). parse_and_validate iterates
# temporal first, so it flags any causal label there as "unknown".
import yaml

PROMPTS_DIR = ROOT / "models" / "llama" / "prompts"

# Per-condition codebook lookup: condition -> {"temporal": set, "causal": set, "joint": set}.
# Pulls every codebook field from each YAML so we can classify a flagged label as:
#   - true hallucination (in NO codebook anywhere)
#   - section_mismatch  (in the OTHER section's codebook for this condition)
#   - other_condition   (only used by a different condition's codebook)
codebooks = {}
for cond in ("temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"):
    cfg = yaml.safe_load((PROMPTS_DIR / f"{cond}.yaml").read_text())
    codebooks[cond] = {
        "temporal": set(cfg.get("allowed_temporal_labels", [])),
        "causal":   set(cfg.get("allowed_causal_labels",   [])),
        "labels":   set(cfg.get("allowed_labels",          [])),    # used by temporal/causal/joint
    }

UNIVERSE = set().union(*[
    cb["temporal"] | cb["causal"] | cb["labels"] for cb in codebooks.values()
])

def classify_unknown(cond: str, label: str) -> str:
    cb = codebooks[cond]
    own = cb["temporal"] | cb["causal"] | cb["labels"]   # everything this condition recognizes
    if label not in UNIVERSE:
        return "true_hallucination"
    if label in own:
        return "section_mismatch"
    return "wrong_condition_label"

hall = audit_df[audit_df["category"] == "label_hallucination"].copy()
hall["hallucinated_label"] = hall["error"].str.extract(r"unknown label: (\S+)")
hall["bucket"] = hall.apply(lambda r: classify_unknown(r["condition"], r["hallucinated_label"]), axis=1)

print(f"{len(hall)} 'unknown label' failures across conditions:\n")
print("By condition × bucket:")
display(hall.groupby(["condition", "bucket"]).size().unstack(fill_value=0))

print("\nLabel × bucket × condition — what's actually getting flagged?")
display(hall.groupby(["condition", "bucket", "hallucinated_label"]).size().to_frame("n").reset_index())

69 'unknown label' failures across conditions:

By condition × bucket:


bucket,section_mismatch,true_hallucination
condition,,
causal,0,1
temporal,0,9
temporal_causal_independent,43,5
temporal_causal_joint,0,11



Label × bucket × condition — what's actually getting flagged?


,condition,bucket,hallucinated_label,n
0,causal,true_hallucination,GIVE,1
1,temporal,true_hallucination,AFTER,5
2,temporal,true_hallucination,CAUSATION,2
3,temporal,true_hallucination,CAUSES,2
4,temporal_causal_independent,section_mismatch,CAUSE,31
5,temporal_causal_independent,section_mismatch,CAUSE_TO_END,11
6,temporal_causal_independent,section_mismatch,ENABLE,1
7,temporal_causal_independent,true_hallucination,AFTER,3
8,temporal_causal_independent,true_hallucination,CAUSES,1
9,temporal_causal_independent,true_hallucination,CAUSE_CHANGE_OF_POSITION_ON_A_SCALE,1


### 6.3 Prose-wrapped JSON (`Expecting value: line 1 column 1`)

LLM prefixed its JSON with conversational text ("Here are the relations:") before the `{`. Affects every condition. Cheap recoverable failure — `parse_and_validate` could strip everything before the first `{`, or the system prompt could be tightened to forbid prose.

In [10]:
# Cases where Llama prefixed the JSON with prose ("Here are the causal relations:") so
# json.loads errors at char 0. Cheap fix: strip everything before first { in parse_and_validate.
prose = audit_df[audit_df["category"] == "prose_wrap"]
print(f"{len(prose)} prose-wrap failures across conditions:")
print(prose.groupby("condition").size().to_string())
print()
print("Output-token distribution — prose-wrap can fire on short OR long outputs:")
display(prose.groupby("condition")["output_tokens"].agg(["count", "min", "median", "max"]).astype(int))
print()
display(prose[["condition", "row_idx", "output_tokens", "n_events", "n_words"]].head(15))

65 prose-wrap failures across conditions:
condition
causal                         11
temporal                        4
temporal_causal_independent    30
temporal_causal_joint          20

Output-token distribution — prose-wrap can fire on short OR long outputs:


,count,min,median,max
condition,,,,
causal,11,664,1613,2048
temporal,4,1891,2048,2048
temporal_causal_independent,30,768,2048,2048
temporal_causal_joint,20,473,1542,2048


,condition,row_idx,output_tokens,n_events,n_words
19,causal,19,997,134,1106
38,causal,38,2048,166,1377
83,causal,83,664,173,1393
101,causal,101,2048,106,717
125,causal,125,1613,128,1058
143,causal,143,2048,335,2662
153,causal,153,1779,142,927
159,causal,159,2048,160,314
160,causal,160,1588,171,1641
162,causal,162,1532,139,605


### 6.4 Other malformed JSON / bad eIDs

Catch-all for structural JSON errors that aren't truncation/prose-wrap (missing delimiters, bad property names) and for `bad eID:` errors from `parse_and_validate` (event IDs referenced in relations that don't exist in the events list). Usually small but worth eyeballing for systematic patterns.

In [11]:
# Structural JSON errors not covered by truncation or prose-wrap — missing delimiters,
# bad property names, etc. Smallest bucket; usually one-off Llama brain farts but worth
# eyeballing to make sure no systematic issue is hiding.
malformed = audit_df[audit_df["category"].isin(["json_malformed", "bad_eid", "other"])]
print(f"{len(malformed)} miscellaneous JSON failures across conditions:")
print(malformed.groupby(["condition", "category"]).size().unstack(fill_value=0))
print()

# Set option to display full column width for errors
pd.set_option('display.max_colwidth', None)
display(malformed[["condition", "row_idx", "category", "error", "n_events", "n_words"]].head(20))
# Reset option if needed for other cells
pd.reset_option('display.max_colwidth')

27 miscellaneous JSON failures across conditions:
category                     bad_eid  json_malformed
condition                                           
causal                             1               6
temporal                           0               5
temporal_causal_independent        0              14
temporal_causal_joint              0               1



,condition,row_idx,category,error,n_events,n_words
28,causal,28,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5484)",85,752
35,causal,35,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5486)",100,746
40,causal,40,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5486)",120,770
84,causal,84,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5482)",81,595
89,causal,89,bad_eid,bad eID: e13|start|Process_start,43,348
126,causal,126,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5478)",74,573
157,causal,157,json_malformed,"Expecting ',' delimiter: line 367 column 1 (char 5492)",103,688
207,temporal_causal_independent,7,json_malformed,Expecting property name enclosed in double quotes: line 95 column 39 (char 5753),96,752
222,temporal_causal_independent,22,json_malformed,Expecting property name enclosed in double quotes: line 97 column 22 (char 5846),63,483
243,temporal_causal_independent,43,json_malformed,Expecting property name enclosed in double quotes: line 95 column 39 (char 5761),101,672
